In [1]:
# ============================================================
# EXPERIMENT 4
# CLEAN START
# ============================================================

from pathlib import Path

import torch
import torch.nn as nn

from ultralytics import YOLO
import ultralytics

print("=" * 70)
print("EXPERIMENT 4 — EcoBotX-Light + CBAM + P2")
print("=" * 70)

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

EXPERIMENT 4 — EcoBotX-Light + CBAM + P2
Ultralytics: 8.4.126
PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
# ============================================================
# ORIGINAL ULTRALYTICS CBAM
# ============================================================

from ultralytics.nn.modules.conv import CBAM

print("=" * 70)
print("ORIGINAL CBAM")
print("=" * 70)

print(CBAM)
print()
print("Constructor:")
print(CBAM.__init__)

ORIGINAL CBAM
<class 'ultralytics.nn.modules.conv.CBAM'>

Constructor:
<function CBAM.__init__ at 0x0000029F96C81620>


In [3]:
# ============================================================
# EcoBotX LAZY CBAM
# ============================================================

class EcoBotXLazyCBAM(nn.Module):
    """
    CBAM whose channel dimension is determined automatically
    from the actual input tensor.

    This avoids the Ultralytics YAML/parser channel mismatch.
    """

    def __init__(self, *args):
        super().__init__()

        # YAML will provide [7].
        # Depending on Ultralytics parser behavior, additional
        # channel arguments may also appear.
        #
        # We only need the kernel size here.

        self.kernel_size = 7

        if len(args) > 0:
            # Find the intended kernel-size argument.
            #
            # For our YAML this will be 7.
            for value in reversed(args):
                if isinstance(value, int) and value in (3, 5, 7):
                    self.kernel_size = value
                    break

        self.cbam = None

    def _build(self, channels, device):
        """
        Build the actual CBAM after seeing the input tensor.
        """

        self.cbam = CBAM(
            channels,
            self.kernel_size
        )

        self.cbam.to(device)

    def forward(self, x):

        # First forward pass:
        # determine actual channel count.
        if self.cbam is None:

            channels = x.shape[1]

            self._build(
                channels,
                x.device
            )

            print(
                f"[CBAM INIT] channels={channels}, "
                f"kernel={self.kernel_size}"
            )

        return self.cbam(x)

In [4]:
# ============================================================
# TEST LAZY CBAM
# ============================================================

print("=" * 70)
print("TESTING LAZY CBAM")
print("=" * 70)

test_module = EcoBotXLazyCBAM(7)

# Simulate P2 feature map
x = torch.randn(
    1,
    32,
    160,
    160
)

with torch.no_grad():
    y = test_module(x)

print()
print("Input shape :", x.shape)
print("Output shape:", y.shape)

assert x.shape == y.shape

print()
print("[OK] Lazy CBAM works with 32 channels")

TESTING LAZY CBAM
[CBAM INIT] channels=32, kernel=7

Input shape : torch.Size([1, 32, 160, 160])
Output shape: torch.Size([1, 32, 160, 160])

[OK] Lazy CBAM works with 32 channels


In [5]:
# ============================================================
# REGISTER MODULE WITH ULTRALYTICS
# ============================================================

import ultralytics.nn.tasks as tasks

tasks.EcoBotXLazyCBAM = EcoBotXLazyCBAM

print("=" * 70)
print("MODULE REGISTRATION")
print("=" * 70)

print(
    "EcoBotXLazyCBAM:",
    tasks.EcoBotXLazyCBAM
)

assert hasattr(
    tasks,
    "EcoBotXLazyCBAM"
)

print("[OK] Module registered")

MODULE REGISTRATION
EcoBotXLazyCBAM: <class '__main__.EcoBotXLazyCBAM'>
[OK] Module registered


In [6]:
# ============================================================
# REGISTER MODULE WITH ULTRALYTICS
# ============================================================

import ultralytics.nn.tasks as tasks

tasks.EcoBotXLazyCBAM = EcoBotXLazyCBAM

print("=" * 70)
print("MODULE REGISTRATION")
print("=" * 70)

print(
    "EcoBotXLazyCBAM:",
    tasks.EcoBotXLazyCBAM
)

assert hasattr(
    tasks,
    "EcoBotXLazyCBAM"
)

print("[OK] Module registered")

MODULE REGISTRATION
EcoBotXLazyCBAM: <class '__main__.EcoBotXLazyCBAM'>
[OK] Module registered


In [7]:
# ============================================================
# EXPERIMENT 4 YAML
# EcoBotX-Light + CBAM + P2
# ============================================================

MODEL_DIR = Path(
    r"G:\EcoBotX_YOLO_training\experiment4"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_YAML = MODEL_DIR / "experiment4_ecobotx_p2.yaml"


yaml_text = r"""
nc: 4

depth_multiple: 0.33
width_multiple: 0.25

backbone:

  # ----------------------------------------------------------
  # P1
  # ----------------------------------------------------------
  - [-1, 1, Conv, [64, 3, 2]]

  # ----------------------------------------------------------
  # P2 / 4
  # ----------------------------------------------------------
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, EcoBotXLazyCBAM, [7]]

  # ----------------------------------------------------------
  # P3 / 8
  # ----------------------------------------------------------
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, EcoBotXLazyCBAM, [7]]

  # ----------------------------------------------------------
  # P4 / 16
  # ----------------------------------------------------------
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, EcoBotXLazyCBAM, [7]]

  # ----------------------------------------------------------
  # P5 / 32
  # ----------------------------------------------------------
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]

  - [-1, 1, SPPF, [1024, 5]]

  - [-1, 1, EcoBotXLazyCBAM, [7]]


head:

  # ==========================================================
  # TOP-DOWN PATH
  # ==========================================================

  # P5 -> P4
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]

  # P4 -> P3
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  # P3 -> P2
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 3], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]

  # P2 attention
  - [-1, 1, EcoBotXLazyCBAM, [7]]

  # ==========================================================
  # BOTTOM-UP PATH
  # ==========================================================

  # P2 -> P3
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 19], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  # P3 -> P4
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 16], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]

  # P4 -> P5
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]

  # ==========================================================
  # FOUR-SCALE DETECTION
  # ==========================================================

  # P2, P3, P4, P5
  - [[21, 24, 27, 30], 1, Detect, [nc]]
"""


MODEL_YAML.write_text(
    yaml_text.strip(),
    encoding="utf-8"
)

print("=" * 70)
print("YAML CREATED")
print("=" * 70)

print(MODEL_YAML)

YAML CREATED
G:\EcoBotX_YOLO_training\experiment4\experiment4_ecobotx_p2.yaml


In [8]:
# ============================================================
# BUILD MODEL ONLY
# ============================================================

print("=" * 70)
print("BUILDING EXPERIMENT 4")
print("=" * 70)

custom_p2_model = YOLO(
    str(MODEL_YAML)
)

print()
print("[OK] MODEL OBJECT CREATED")

BUILDING EXPERIMENT 4
[CBAM INIT] channels=32, kernel=7
[CBAM INIT] channels=64, kernel=7
[CBAM INIT] channels=128, kernel=7
[CBAM INIT] channels=256, kernel=7
[CBAM INIT] channels=32, kernel=7

[OK] MODEL OBJECT CREATED


In [9]:
# ============================================================
# MODEL INFORMATION
# ============================================================

print("=" * 70)
print("EXPERIMENT 4 MODEL INFORMATION")
print("=" * 70)

custom_p2_model.info()

EXPERIMENT 4 MODEL INFORMATION
experiment4_ecobotx_p2 summary: 186 layers, 3,359,194 parameters, 3,359,178 gradients, 22.9 GFLOPs


(186, 3359194, 3359178, 22.897183552)

In [10]:
# ============================================================
# EXPERIMENT 4 — MODEL SANITY CHECK
# ============================================================

import torch

print("=" * 70)
print("EXPERIMENT 4 — MODEL SANITY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check model parameters
# ------------------------------------------------------------

total_params = sum(p.numel() for p in custom_p2_model.parameters())
trainable_params = sum(
    p.numel() for p in custom_p2_model.parameters()
    if p.requires_grad
)

print(f"\nTotal parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

# ------------------------------------------------------------
# 2. Forward pass
# ------------------------------------------------------------

print("\nRunning dummy forward pass...")

device = "cuda" if torch.cuda.is_available() else "cpu"

custom_p2_model.to(device)

dummy_input = torch.zeros(
    1, 3, 640, 640,
    device=device
)

with torch.no_grad():
    output = custom_p2_model.model(dummy_input)

print("[OK] Forward pass successful.")

# ------------------------------------------------------------
# 3. Device
# ------------------------------------------------------------

print(f"\nDevice: {device}")

# ------------------------------------------------------------
# 4. Model info
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SANITY CHECK PASSED")
print("=" * 70)

EXPERIMENT 4 — MODEL SANITY CHECK

Total parameters     : 3,359,194
Trainable parameters : 3,359,178

Running dummy forward pass...
[OK] Forward pass successful.

Device: cuda

SANITY CHECK PASSED


In [11]:
# ============================================================
# EXPERIMENT 4 — TRAINING
# EcoBotX-Light + CBAM + P2
# ============================================================

from pathlib import Path
import torch

print("=" * 70)
print("EXPERIMENT 4 — TRAINING")
print("EcoBotX-Light + CBAM + P2")
print("=" * 70)

print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

DATASET = r"G:\EcoBotX_YOLO\dataset.yaml"

PROJECT = r"G:\EcoBotX_YOLO_training\experiment4_results"
NAME = "EcoBotX_Light_CBAM_P2"

print("\nDataset:")
print(DATASET)

print("\nTraining configuration:")
print("Epochs      : 100")
print("Image size  : 640")
print("Batch size  : 8")
print("Device      : 0")
print("Pretrained  : False")
print("Optimizer   : auto")
print("Patience    : 20")

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

results = custom_p2_model.train(
    data=DATASET,

    epochs=100,
    imgsz=640,
    batch=8,

    device=0,
    workers=4,

    pretrained=False,

    optimizer="auto",

    patience=20,

    project=PROJECT,
    name=NAME,

    save_period=10,

    plots=True,

    verbose=True
)

print("\n" + "=" * 70)
print("EXPERIMENT 4 TRAINING FINISHED")
print("=" * 70)

EXPERIMENT 4 — TRAINING
EcoBotX-Light + CBAM + P2
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU

Dataset:
G:\EcoBotX_YOLO\dataset.yaml

Training configuration:
Epochs      : 100
Image size  : 640
Batch size  : 8
Device      : 0
Pretrained  : False
Optimizer   : auto
Patience    : 20
New https://pypi.org/project/ultralytics/8.4.128 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.126  Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=G:\EcoBotX_YOLO\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=

KeyboardInterrupt: 